In [1]:
import sys

print(sys.version)
print(sys.executable)

3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]
C:\Users\lenovo\Desktop\Agentic-RAG-Research-Assistant\rag_env\Scripts\python.exe


In [2]:
import fitz
from pathlib import Path

print("PyMuPDF version:", fitz.version)


PyMuPDF version: ('1.28.2', '1.28.2', None)


In [3]:
pdf_path = Path("C:/Users/lenovo/Desktop/Agentic-RAG-Research-Assistant/data/research_papers/r1.pdf")

print("File exists:", pdf_path.exists())
print("File:", pdf_path)

File exists: True
File: C:\Users\lenovo\Desktop\Agentic-RAG-Research-Assistant\data\research_papers\r1.pdf


In [4]:
doc = fitz.open(pdf_path)

print("Number of pages:", len(doc))

Number of pages: 19


In [5]:
text = ""

for page in doc:
    text += page.get_text()

print("Total characters:", len(text))

Total characters: 69078


In [6]:
print("Total characters:", len(text))
print("Total words:", len(text.split()))

Total characters: 69078
Total words: 9886


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

C:\Users\lenovo\Desktop\Agentic-RAG-Research-Assistant\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
chunks = text_splitter.split_text(text)

print("Number of chunks:", len(chunks))

Number of chunks: 88


In [9]:
print(chunks[0])

Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks
Patrick Lewis†‡, Ethan Perez⋆,
Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,
Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†
†Facebook AI Research; ‡University College London; ⋆New York University;
plewis@fb.com
Abstract
Large pre-trained language models have been shown to store factual knowledge
in their parameters, and achieve state-of-the-art results when ﬁne-tuned on down-
stream NLP tasks. However, their ability to access and precisely manipulate knowl-
edge is still limited, and hence on knowledge-intensive tasks, their performance
lags behind task-speciﬁc architectures. Additionally, providing provenance for their
decisions and updating their world knowledge remain open research problems. Pre-
trained models with a differentiable access mechanism to explicit non-parametric


In [10]:
chunk_lengths = [len(chunk) for chunk in chunks]

print("Smallest chunk:", min(chunk_lengths))
print("Largest chunk:", max(chunk_lengths))
print("Average chunk:", sum(chunk_lengths) / len(chunk_lengths))

Smallest chunk: 902
Largest chunk: 998
Average chunk: 956.0


In [11]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=chunk,
        metadata={
            "source": "rag_paper.pdf",
            "chunk_id": i
        }
    )
    for i, chunk in enumerate(chunks)
]

print("Documents created:", len(documents))

Documents created: 88


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

In [13]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 355.01it/s]


In [14]:
test_embedding = embedding_model.embed_query(
    "What is retrieval augmented generation?"
)

print("Embedding dimensions:", len(test_embedding))

Embedding dimensions: 384


In [15]:
from langchain_community.vectorstores import FAISS

C:\Users\lenovo\AppData\Local\Temp\ipykernel_1432\1059762981.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [16]:
vector_store = FAISS.from_documents(
    documents,
    embedding_model
)

In [17]:
vector_store.save_local("C:/Users/lenovo/Desktop/Agentic-RAG-Research-Assistant/vector_store/r1_index")

In [18]:
query = "What is Retrieval-Augmented Generation?"

results = vector_store.similarity_search(
    query,
    k=3
)

In [19]:
for i, result in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(result.page_content[:1000])


--- Result 1 ---
pieces of retrieved content, as well as learning latent retrieval, and retrieving evidence documents
rather than related training pairs. This said, RAG techniques may work well in these settings, and
could represent promising future work.
6
Discussion
In this work, we presented hybrid generation models with access to parametric and non-parametric
memory. We showed that our RAG models obtain state of the art results on open-domain QA. We
found that people prefer RAG’s generation over purely parametric BART, ﬁnding RAG more factual
and speciﬁc. We conducted an thorough investigation of the learned retrieval component, validating
its effectiveness, and we illustrated how the retrieval index can be hot-swapped to update the model
without requiring any retraining. In future work, it may be fruitful to investigate if the two components
can be jointly pre-trained from scratch, either with a denoising objective similar to BART or some

--- Result 2 ---
point precision to mana

In [20]:
query = "What are the limitations of traditional language models?"

In [21]:
results = vector_store.similarity_search(
    query,
    k=3
)

for i, result in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(result.page_content[:1000])


--- Result 1 ---
[46] Ethan Perez, Siddharth Karamcheti, Rob Fergus, Jason Weston, Douwe Kiela, and Kyunghyun
Cho. Finding generalizable evidence by learning to convince q&a models. In Proceedings
of the 2019 Conference on Empirical Methods in Natural Language Processing and the 9th
International Joint Conference on Natural Language Processing (EMNLP-IJCNLP), pages
2402–2411, Hong Kong, China, November 2019. Association for Computational Linguistics.
doi: 10.18653/v1/D19-1244. URL https://www.aclweb.org/anthology/D19-1244.
[47] Fabio Petroni, Tim Rocktäschel, Sebastian Riedel, Patrick Lewis, Anton Bakhtin, Yuxiang Wu,
and Alexander Miller. Language models as knowledge bases? In Proceedings of the 2019
Conference on Empirical Methods in Natural Language Processing and the 9th International
Joint Conference on Natural Language Processing (EMNLP-IJCNLP), pages 2463–2473, Hong
Kong, China, November 2019. Association for Computational Linguistics. doi: 10.18653/v1/

--- Result 2 ---
N16-10

In [22]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

print("API key loaded:", api_key is not None)

API key loaded: True


In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [25]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

In [26]:
response = llm.invoke(
    "Explain Retrieval-Augmented Generation in three sentences."
)

print(response.content)

C:\Users\lenovo\Desktop\Agentic-RAG-Research-Assistant\rag_env\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Retrieval-Augmented Generation (RAG) is an AI technique that improves the accuracy of a language model by connecting it to an external knowledge base. When a user submits a prompt, the system retrieves relevant information from private or up-to-date documents and passes that context to the model to generate a response. By grounding the model\'s output in real-time data, RAG reduces false information ("hallucinations") and provides verifiable, source-backed answers.', 'extras': {'signature': 'EpgZCpUZARFNMg98uVh28/9wKTfSynu9qCW0B8xlWIb6VzLGpaXnjBRfd74rZ7eHv/V/KncbvH/NOD9MOBa9Xm+lVImmsL9l/vziJDwBKF8PK1tGpJ4sFf5pSnYHEwdakUEIZQWDKaiAuxe4GzltutkEVpcFe9EsfN51IlP1x5LL2e/KWh3fRjDZHBfZtywRJWQPkUcTwk07OS55Ef3fJfLIvnpZRNlbd/t2KvNqDoTzghcM+qzJdWJfb7NPWJR4vJnX0qev1tGc49esY/pKU13kh8zkFel5cggulhuwZnBhRz5lywetVLweRjUrCrchTtnPmp5knCQOTtWlGj4Pla3UQijXa+tjhgT0/WVoPOUdV+/PXcgs/WfKbGgVhIiCqo5e5tRXTODzJ4Tm8DseloDJw6jIvyEpE9vJmkiT22VoeSBkHYnnrJPgmBYUKn7PVheRwoxV0SyjCrHepnmuRUX+2a/O

In [27]:
query = "What is Retrieval-Augmented Generation?"

retrieved_docs = vector_store.similarity_search(
    query,
    k=3
)

print("Retrieved documents:", len(retrieved_docs))

Retrieved documents: 3


In [28]:
for i, doc in enumerate(retrieved_docs):

    print(f"\n--- Result {i+1} ---")
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print("Chunk:", doc.metadata.get("chunk_id"))
    print("\n", doc.page_content[:700])


--- Result 1 ---
Source: rag_paper.pdf
Page: None
Chunk: 44

 pieces of retrieved content, as well as learning latent retrieval, and retrieving evidence documents
rather than related training pairs. This said, RAG techniques may work well in these settings, and
could represent promising future work.
6
Discussion
In this work, we presented hybrid generation models with access to parametric and non-parametric
memory. We showed that our RAG models obtain state of the art results on open-domain QA. We
found that people prefer RAG’s generation over purely parametric BART, ﬁnding RAG more factual
and speciﬁc. We conducted an thorough investigation of the learned retrieval component, validating
its effectiveness, and we illustrated how the retrieval index can

--- Result 2 ---
Source: rag_paper.pdf
Page: None
Chunk: 87

 point precision to manage memory and disk footprints.
H
Retrieval Collapse
In preliminary experiments, we observed that for some tasks such as story generation [11], the
ret

In [29]:
context = "\n\n".join(
    [
        f"[Source {i+1} | Page {doc.metadata.get('page')}]\n{doc.page_content}"
        for i, doc in enumerate(retrieved_docs)
    ]
)

print(context)

[Source 1 | Page None]
pieces of retrieved content, as well as learning latent retrieval, and retrieving evidence documents
rather than related training pairs. This said, RAG techniques may work well in these settings, and
could represent promising future work.
6
Discussion
In this work, we presented hybrid generation models with access to parametric and non-parametric
memory. We showed that our RAG models obtain state of the art results on open-domain QA. We
found that people prefer RAG’s generation over purely parametric BART, ﬁnding RAG more factual
and speciﬁc. We conducted an thorough investigation of the learned retrieval component, validating
its effectiveness, and we illustrated how the retrieval index can be hot-swapped to update the model
without requiring any retraining. In future work, it may be fruitful to investigate if the two components
can be jointly pre-trained from scratch, either with a denoising objective similar to BART or some

[Source 2 | Page None]
point precis

In [30]:
prompt = f"""
You are an academic research assistant.

Answer the user's question using ONLY the research
evidence provided below.

Rules:
1. Do not invent information.
2. If the evidence is insufficient, say so.
3. Base factual claims on the provided evidence.
4. Include the source page when making important claims.

Research Evidence:

{context}

User Question:

{query}

Answer:
"""

In [31]:
response = llm.invoke(prompt)

print(response.content)

C:\Users\lenovo\Desktop\Agentic-RAG-Research-Assistant\rag_env\Lib\site-packages\langchain_google_genai\chat_models.py:3610: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Based on the provided research evidence, Retrieval-Augmented Generation (RAG) refers to a class of hybrid generation models that utilize both parametric and non-parametric memory (Source 1, Page None). \n\nKey details about RAG provided in the text include:\n\n* **Structure & Components:** RAG combines a generator (such as BART) with a learned, dense retrieval mechanism/component that retrieves relevant evidence documents to assist with tasks (Source 1, Page None; Source 3, Page None). Known model variants include *RAG-Sequence* and *RAG-Token* (Source 3, Page None).\n* **Flexibility:** The retrieval index can be "hot-swapped" to update the model\'s knowledge without requiring any retraining (Source 1, Page None).\n* **Performance:** RAG models achieve state-of-the-art results on open-domain QA (Source 1, Page None). Compared to purely parametric models like BART, RAG generates content that human evaluators find more factual, specific, and significantly more 